# Slead 3a — Fast Station-Only Forward Model (keeps your variables)

In [1]:
import os, h5py, numpy as np
import utm
from pyproj import Proj, transform
try:
    from VSM_forward import okada as OKADA
    print("Using VSM_forward.okada")
except Exception:
    OKADA = None
    print("WARNING: Using stub okada (replace with VSM_forward.okada for real results)")    


Using VSM_forward.okada


In [2]:
def _okada_stub(x, y, xtlc, ytlc, dtlc, length, width, strike, dip, ss, ds, slip, rake, nu):
    xv = x - xtlc; yv = y - ytlc
    r2 = xv*xv + yv*yv + (0.3*length)**2
    fac = slip / (r2 + 1e-6)
    return fac * xv, fac * yv, fac * (xv*yv / (r2+1e-6))
if OKADA is None:
    OKADA = _okada_stub


In [3]:
# 1) Load model3a.h5 and reconstruct `patches` exactly like your notebook
with h5py.File('../Data/model3a.h5','r') as f:
    m          = f['m'][:]              # (n_patches,)
    rectangles = f['rectangles'][:]     # (n_patches, 4, 3)
    lat        = f['lat'][:]            # (Ny, Nx)
    lon        = f['lon'][:]            # (Ny, Nx)

patches = []
for rect in rectangles:
    xtlc, ytlc, dtlc = rect[0]
    length = np.linalg.norm(rect[1][:2] - rect[0][:2])
    width  = np.linalg.norm(rect[3][:2] - rect[0][:2])
    dx, dy = rect[1][0] - rect[0][0], rect[1][1] - rect[0][1]
    strike = np.degrees(np.arctan2(dy, dx))
    dz = rect[3][2] - rect[0][2]
    horiz = np.linalg.norm(rect[3][:2] - rect[0][:2])
    dip    = np.degrees(np.arctan2(-dz, horiz))
    patches.append({'xtlc': xtlc, 'ytlc': ytlc, 'dtlc': dtlc,
                    'length': length, 'width': width,
                    'strike': strike, 'dip': dip})
print(f"Loaded {len(patches)} patches")


Loaded 3484 patches


In [4]:
# 2) Build X,Y grids (names preserved), but we won't compute fields on the grid
wgs84    = Proj('epsg:4326')
utm_proj = Proj(proj='utm', zone=10, northern=True, ellps='WGS84')
lon_f = lon.flatten()
lat_f = lat.flatten()
x_f, y_f = transform(wgs84, utm_proj, lon_f, lat_f)
X = x_f.reshape(lat.shape)
Y = y_f.reshape(lat.shape)
print("X/Y shapes:", X.shape, Y.shape)


/tmp/ipykernel_1267022/2680319899.py:6: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-from-pyproj-1
  x_f, y_f = transform(wgs84, utm_proj, lon_f, lat_f)


X/Y shapes: (1152, 1021) (1152, 1021)


In [5]:
# 3) Reuse your station variables if present; else define defaults
try:
    stations  # from your session
    print("Using pre-existing `stations` dict.")
except NameError:
    stations = {
        '2501': (45.95480372, -130.0090668),
        '2502': (45.96257971, -129.9910622),
        '2503': (45.94700845, -130.0265777),
        '2504': (45.95882833, -130.0114997),
        'NodeF':(45.95485,-130.008772),
        'NodeE':(45.939888,-129.974113),
        'NodeB':(45.933585,-130.013857),
    }
    print("Defined default `stations`")

station_utm = {}
for fid, (lat_s, lon_s) in stations.items():
    e, n, _, _ = utm.from_latlon(lat_s, lon_s)
    station_utm[fid] = (e, n)
print("station_utm:", station_utm)


Defined default `stations`
station_utm: {'2501': (421802.15868199023, 5089520.875157327), '2502': (423208.1678999553, 5090367.323655002), '2503': (420433.9977720107, 5088672.108341966), '2504': (421619.2962082646, 5089970.421139045), 'NodeF': (421825.0692740191, 5089525.727926807), 'NodeE': (424490.6507644805, 5087829.959733143), 'NodeB': (421400.9591348714, 5087168.075465049)}


In [6]:
stations

{'2501': (45.95480372, -130.0090668),
 '2502': (45.96257971, -129.9910622),
 '2503': (45.94700845, -130.0265777),
 '2504': (45.95882833, -130.0114997),
 'NodeF': (45.95485, -130.008772),
 'NodeE': (45.939888, -129.974113),
 'NodeB': (45.933585, -130.013857)}

In [7]:
# 4) Station-only forward model: fills Ustat (Ux,Uy,Uz at each station)
fids = list(station_utm.keys())
e_s  = np.array([station_utm[f][0] for f in fids])
n_s  = np.array([station_utm[f][1] for f in fids])

Ux_s = np.zeros_like(e_s, dtype=np.float32)
Uy_s = np.zeros_like(e_s, dtype=np.float32)
Uz_s = np.zeros_like(e_s, dtype=np.float32)

for j, p in enumerate(patches):
    ux, uy, uz = OKADA(
        e_s, n_s,
        p['xtlc'], p['ytlc'], -p['dtlc'],
        p['length'], p['width'],
        p['strike'], p['dip'],
        0.0, 0.0, float(m[j]), 'R', 0.25
    )
    Ux_s += ux.astype(np.float32)
    Uy_s += uy.astype(np.float32)
    Uz_s += uz.astype(np.float32)

Ustat = {fid: np.array([Ux_s[i], Uy_s[i], Uz_s[i]], dtype=np.float32) for i, fid in enumerate(fids)}
print("Ustat:")
for k,v in Ustat.items():
    print(k, v)


Ustat:
2501 [ 0.00550172 -0.02801009  1.2680349 ]
2502 [0.3178493  0.05797915 0.80562806]
2503 [-0.2908562  -0.1480584   0.80975866]
2504 [-0.02458345  0.00206377  1.2698818 ]
NodeF [ 0.01252516 -0.02628824  1.2676758 ]
NodeE [0.20024401 0.05387427 0.5210902 ]
NodeB [-0.20616828 -0.20243496  0.7977739 ]


In [8]:
# 5) Your baseline projection helper — unchanged semantics
def predict_range_change(fidA, fidB):
    eA, nA = station_utm[fidA]
    eB, nB = station_utm[fidB]
    baseline = np.array([eB-eA, nB-nA], dtype=np.float64)
    L = np.hypot(baseline[0], baseline[1])
    if L == 0:
        return 0.0
    u_hat = baseline / L
    dU = Ustat[fidB][:2] - Ustat[fidA][:2]
    return float(np.dot(dU, u_hat))

for A, B in [('2503','2504'),('2502','2504'),('2502','2503')]:
    print(f"Predicted Δrange {A}→{B}: {predict_range_change(A,B):.4f} m")


Predicted Δrange 2503→2504: 0.2904 m
Predicted Δrange 2502→2504: 0.3458 m
Predicted Δrange 2502→2503: 0.6268 m
